# **Análisis de Terremotos en Chile (2000-2024)**
## **Fase 3: Programación Orientada a Objetos**

**Nombres**: Ademir Ortiz, Bruno Sepúlveda Manrique, Lucas Bravo  
**Grupo**: 9  
**Curso**: Programación para la Ciencia de Datos  
**Profesor**: Omar Salinas Sanchez

## **1. Introducción: Arquitectura POO**

Este notebook implementa un sistema completo de análisis sísmico utilizando **Programación Orientada a Objetos**.

### Principios SOLID Implementados:
- **S**ingle Responsibility: Cada clase tiene una responsabilidad única
- **O**pen/Closed: Abierto para extensión, cerrado para modificación
- **L**iskov Substitution: Las subclases pueden sustituir a la clase base
- **I**nterface Segregation: Interfaces específicas y cohesivas
- **D**ependency Inversion: Dependencia de abstracciones

### Patrones de Diseño:
1. **Abstract Factory**: Clase Transformer base
2. **Template Method**: Método ejecutar()
3. **Strategy Pattern**: CategorizadorDatos
4. **Builder Pattern**: PipelineProcesamiento
5. **Composite Pattern**: Agregación de transformadores

## **2. Importación de Librerías**

In [70]:
# Manipulación de datos
import pandas as pd
import numpy as np
import time
import logging
import re
from datetime import datetime
from typing import List, Dict, Tuple, Optional
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

## Configuracion del Dataset


In [71]:
# Ruta global del dataset
RUTA_DATASET = r'../../data/raw/EarthquakesChile_2000-2024.csv'
print(f'Dataset configurado: {RUTA_DATASET}')

Dataset configurado: ../../data/raw/EarthquakesChile_2000-2024.csv


## **3. Clase Base Transformer (Abstract Factory Pattern)**

Clase abstracta que define el contrato para todas las transformaciones.
Implementa Template Method Pattern en el método `ejecutar()`.

In [72]:
class Transformer(ABC):
    """
    Clase base abstracta para transformaciones de datos.
    Attributes:
        nombre (str): Identificador del transformador
        documentacion (str): Descripción funcional
        _logger: Logger para trazabilidad
        _tiempo_ejecucion: Métrica de performance
    """
    def __init__(self, nombre: str, documentacion: str = ""):
        self.nombre = nombre
        self.documentacion = documentacion
        self._logger = logging.getLogger(f"Transformer.{nombre}")
        self._tiempo_ejecucion = None
    @abstractmethod
    def validar_entrada(self, df: pd.DataFrame) -> bool:
        """Valida precondiciones del DataFrame"""
        pass
    @abstractmethod
    def transformar(self, df: pd.DataFrame) -> pd.DataFrame:
        """Aplica la transformación"""
        pass
    def ejecutar(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Template Method: Define el flujo de ejecución.
        1. Validación
        2. Transformación
        3. Logging
        """
        inicio = time.time()
        self._logger.info(f"Iniciando: {self.nombre}")
        if not self.validar_entrada(df):
            raise ValueError(f"Validación fallida en {self.nombre}")
        df_resultado = self.transformar(df)
        self._tiempo_ejecucion = time.time() - inicio
        self._logger.info(f"Completado en {self._tiempo_ejecucion:.4f}s")
        return df_resultado
    def get_tiempo_ejecucion(self) -> Optional[float]:
        return self._tiempo_ejecucion
    def __repr__(self) -> str:
        return f"{self.__class__.__name__}('{self.nombre}')"

## **4. DataClasses para Resultados**

Encapsulación de resultados y configuraciones.

In [73]:
@dataclass
class ResultadoTransformacion:
    """Encapsula resultado de una transformación"""
    transformador: str
    estado: str
    filas_entrada: int
    filas_salida: int
    columnas_entrada: int
    columnas_salida: int
    tiempo_ejecucion: float
    timestamp: datetime = field(default_factory=datetime.now)
    def __str__(self) -> str:
        return (f"Transformador: {self.transformador}\n"
                f"  Estado: {self.estado}\n"
                f"  Filas: {self.filas_entrada} → {self.filas_salida}\n"
                f"  Columnas: {self.columnas_entrada} → {self.columnas_salida}\n"
                f"  Tiempo: {self.tiempo_ejecucion:.4f}s")
@dataclass
class ConfiguracionCategoria:
    """Configuración para categorización"""
    columna: str
    bins: List[float]
    etiquetas: List[str]
    descripcion: str = ""
    def validar(self) -> bool:
        return len(self.bins) == len(self.etiquetas) + 1

In [74]:
# FUNCIÓN DE UTILIDAD: Ejecutor con Logging Detallado
def ejecutar_transformador_con_log(transformador, df_entrada, nombre_prueba=""):
    """
    Ejecuta un transformador con logging detallado de todas las etapas.
    
    Parámetros:
        transformador: Instancia del transformador a ejecutar
        df_entrada: DataFrame de entrada
        nombre_prueba: Nombre descriptivo para la prueba (opcional)
    
    Retorna:
        DataFrame transformado
    
    Muestra:
        1. Validación de entrada
        2. Transformaciones realizadas
        3. Comparativa antes/después
        4. Tiempo de ejecución
    """
    print("\n" + "=" * 100)
    print(f"TRANSFORMADOR: {nombre_prueba or transformador.nombre}")
    print("=" * 100)
    
    # PASO 1: Información de entrada
    print(f"\n📥 ENTRADA:")
    print(f"   Filas: {df_entrada.shape[0]:,}")
    print(f"   Columnas: {df_entrada.shape[1]}")
    print(f"   Columnas presentes: {df_entrada.columns.tolist()[:5]}...")  # Primeras 5
    print(f"   Memoria: {df_entrada.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # PASO 2: Validación
    print(f"\n✓ VALIDANDO entrada...")
    try:
        es_valida = transformador.validar_entrada(df_entrada)
        if es_valida:
            print(f"   ✅ Validación EXITOSA - DataFrame cumple con precondiciones")
        else:
            print(f"   ❌ Validación FALLIDA")
            return df_entrada
    except Exception as e:
        print(f"   ❌ Error en validación: {e}")
        return df_entrada
    
    # PASO 3: Capturar estado antes
    columnas_antes = set(df_entrada.columns)
    filas_antes = len(df_entrada)
    df_antes_copia = df_entrada.copy()
    
    # PASO 4: Ejecutar transformación
    print(f"\n⚙️  TRANSFORMANDO...")
    try:
        df_resultado = transformador.ejecutar(df_entrada)
        tiempo_exec = transformador.get_tiempo_ejecucion()
        print(f"   ✅ Transformación EXITOSA en {tiempo_exec:.4f} segundos")
    except Exception as e:
        print(f"   ❌ Error en transformación: {e}")
        import traceback
        traceback.print_exc()
        return df_entrada
    
    # PASO 5: Analizar cambios
    print(f"\n📊 CAMBIOS REALIZADOS:")
    
    # Cambios en dimensiones
    filas_despues = len(df_resultado)
    cambio_filas = filas_despues - filas_antes
    print(f"   Filas: {filas_antes:,} → {filas_despues:,} ({'+' if cambio_filas > 0 else ''}{cambio_filas:,})")
    
    # Cambios en columnas
    columnas_despues = set(df_resultado.columns)
    columnas_nuevas = columnas_despues - columnas_antes
    columnas_eliminadas = columnas_antes - columnas_despues
    
    if columnas_nuevas:
        print(f"   ✨ Columnas NUEVAS ({len(columnas_nuevas)}): {sorted(list(columnas_nuevas))[:5]}...")
    if columnas_eliminadas:
        print(f"   🗑️  Columnas ELIMINADAS ({len(columnas_eliminadas)}): {sorted(list(columnas_eliminadas))[:5]}...")
    if not columnas_nuevas and not columnas_eliminadas:
        print(f"   ↔️  Estructura de columnas SIN CAMBIOS")
    
    print(f"   Columnas totales: {len(columnas_antes)} → {len(columnas_despues)}")
    
    # PASO 6: Muestreo de datos
    print(f"\n📋 MUESTREO DE DATOS (primeras 3 filas):")
    
    # Mostrar columnas relevantes
    if columnas_nuevas:
        print(f"\n   📍 DATOS NUEVOS GENERADOS:")
        cols_mostrar = list(columnas_nuevas)[:3]  # Limitar a 3
        try:
            print(df_resultado[cols_mostrar].head(3).to_string())
        except:
            print(df_resultado[cols_mostrar].head(3))
    else:
        print(f"\n   📍 TRANSFORMACIÓN APLICADA:")
        cols_mostrar = list(columnas_antes)[:3]
        try:
            print(df_resultado[cols_mostrar].head(3).to_string())
        except:
            print(df_resultado[cols_mostrar].head(3))
    
    # PASO 7: Estadísticas
    print(f"\n📈 ESTADÍSTICAS:")
    print(f"   Memoria: {df_antes_copia.memory_usage(deep=True).sum() / 1024**2:.2f} MB → {df_resultado.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"   Valores nulos: {df_antes_copia.isnull().sum().sum():,} → {df_resultado.isnull().sum().sum():,}")
    
    print("=" * 100 + "\n")
    
    return df_resultado

## 📋 Función de Utilidad: `ejecutar_transformador_con_log()`

### Propósito
Esta función ejecuta cualquier transformador mostrando **todas las etapas del proceso** con evidencia por pantalla:

1. **✓ VALIDACIÓN** → Comprueba que la entrada sea válida (precondiciones)
2. **⚙️ TRANSFORMACIÓN** → Ejecuta la lógica de transformación
3. **📊 ANÁLISIS DE CAMBIOS** → Muestra qué cambió:
   - Número de filas (eliminadas/agregadas)
   - Columnas nuevas creadas
   - Columnas eliminadas
4. **📋 MUESTREO** → Muestra datos antes/después
5. **📈 ESTADÍSTICAS** → Memoria, valores nulos, etc.

### Parámetros
```python
ejecutar_transformador_con_log(
    transformador,      # Instancia del transformador (ej: CargadorDatos())
    df_entrada,        # DataFrame a transformar
    nombre_prueba=""   # Nombre descriptivo (opcional)
)
```

### Ejemplo de Uso
```python
# Ejemplo 1: Con nombre descriptivo
df_resultado = ejecutar_transformador_con_log(
    ProcesadorFechaUTC(),
    df_datos,
    nombre_prueba="ProcesadorFechaUTC - Extrae ANIO_UTC, MES_UTC, DIA_UTC"
)

# Ejemplo 2: Sin nombre (usa nombre del transformador)
df_resultado = ejecutar_transformador_con_log(
    RenombradoVariables(),
    df_datos
)
```

## **5. Transformadores Básicos**

### 5.1 Cargador de Datos

In [83]:
class CargadorDatos(Transformer):
    """
    Carga datos desde CSV con validación de integridad.
    Responsabilidades:
    - Lectura de archivo
    - Validación de formato
    - Manejo de errores
    """
    def __init__(self, ruta: str, encoding: str = 'utf-8'):
        super().__init__("CargadorDatos", "Carga datos desde CSV")
        self.ruta = ruta
        self.encoding = encoding
    def validar_entrada(self, df: pd.DataFrame) -> bool:
        return True
    def transformar(self, df: pd.DataFrame) -> pd.DataFrame:
        try:
            df_cargado = pd.read_csv(self.ruta, encoding=self.encoding)
            self._logger.info(f"Cargadas {df_cargado.shape[0]} filas, {df_cargado.shape[1]} columnas")
            return df_cargado
        except FileNotFoundError:
            self._logger.error(f"Archivo no encontrado: {self.ruta}")
            raise
        except Exception as e:
            self._logger.error(f"Error al cargar: {str(e)}")
            raise

In [76]:
# PRUEBA 1: Cargador de Datos
print("\n" + "="*100)
print("PRUEBA 1: CargadorDatos")
print("="*100)
try:
    cargador = CargadorDatos(RUTA_DATASET)
    # Para el cargador, pasamos un DataFrame vacío como entrada (es el primer paso)
    df_cargado = ejecutar_transformador_con_log(
        cargador, 
        pd.DataFrame(), 
        nombre_prueba="CargadorDatos - Carga desde CSV"
    )
    print(f"✓ Datos cargados y listos para siguiente transformación")
except Exception as e:
    print(f"✗ Error crítico: {e}")
    import traceback
    traceback.print_exc()

2026-06-19 16:21:19,173 - INFO - Iniciando: CargadorDatos



PRUEBA 1: CargadorDatos

TRANSFORMADOR: CargadorDatos - Carga desde CSV

📥 ENTRADA:
   Filas: 0
   Columnas: 0
   Columnas presentes: []...
   Memoria: 0.00 MB

✓ VALIDANDO entrada...
   ✅ Validación EXITOSA - DataFrame cumple con precondiciones

⚙️  TRANSFORMANDO...


2026-06-19 16:21:19,415 - INFO - Cargadas 134062 filas, 8 columnas
2026-06-19 16:21:19,416 - INFO - Completado en 0.2428s


   ✅ Transformación EXITOSA en 0.2428 segundos

📊 CAMBIOS REALIZADOS:
   Filas: 0 → 134,062 (+134,062)
   ✨ Columnas NUEVAS (8): ['Date', 'Hour', 'Latitude', 'Location', 'Longitude']...
   Columnas totales: 0 → 8

📋 MUESTREO DE DATOS (primeras 3 filas):

   📍 DATOS NUEVOS GENERADOS:
       Hour Profoundity        Date
0  13:35:22      243 km  2024-03-01
1  12:41:03      259 km  2024-03-01
2  05:27:13       26 km  2024-03-01

📈 ESTADÍSTICAS:
   Memoria: 0.00 MB → 49.23 MB
   Valores nulos: 0.0 → 3,304

✓ Datos cargados y listos para siguiente transformación


### 5.2 Limpieza de Duplicados

In [77]:
# Ruta del dataset
RUTA_DATASET = r'../../data/raw/EarthquakesChile_2000-2024.csv'
# Construcción del pipeline usando Builder Pattern
print("="*70)
print("="*70)
pipeline = PipelineProcesamiento()
# API Fluida: Cargar datos
pipeline.cargar_datos(RUTA_DATASET)
# 8.2) Agregar transformadores básicos
pipeline.agregar_transformador(LimpiezaDuplicados())
# 8.1) Renombrado de variables
pipeline.agregar_transformador(RenombradoVariables())
# 8.3) Procesamiento de fecha UTC
pipeline.agregar_transformador(ProcesadorFechaUTC())
# 8.4) Separación de profundidad
pipeline.agregar_transformador(SeparadorProfundidad())
# 8.6) Separación de magnitud
pipeline.agregar_transformador(SeparadorMagnitud())
# 8.5) Separación de ubicación
pipeline.agregar_transformador(SeparadorUbicacion())
# 8.7) Estandarización de magnitud a MW
pipeline.agregar_transformador(EstandarizadorMagnitudMW())
# 8.8) Filtro geográfico (solo eventos en Chile)
pipeline.agregar_transformador(FiltroGeograficoChile())
# 8.11) Tratamiento de valores nulos
pipeline.agregar_transformador(TratamientoNulos())
# 8.13) Configurar categorizador con Strategy Pattern
categorizador = CategorizadorDatos()
# Categorización de profundidad (somero, intermedio, profundo)
categorizador.registrar_categoria(
    nombre='CATEGORIA_PROFUNDIDAD',
    columna='PROFUNDIDAD_VALOR',
    bins=[0, 70, 300, float('inf')],
    etiquetas=['somero', 'intermedio', 'profundo'],
    descripcion='Categorización de profundidad sísmica'
)
# Categorización de magnitud (Micro, Menor, Ligero, Moderado, Fuerte, Mayor, Grande)
categorizador.registrar_categoria(
    nombre='CATEGORIA_MAGNITUD',
    columna='MAGNITUD_MW',
    bins=[0, 2, 4, 5, 6, 7, 8, float('inf')],
    etiquetas=['Micro', 'Menor', 'Ligero', 'Moderado', 'Fuerte', 'Mayor', 'Grande'],
    descripcion='Categorización de magnitud sísmica'
)
# Categorización de distancia al epicentro
categorizador.registrar_categoria(
    nombre='CATEGORIA_DISTANCIA',
    columna='DISTANCIA_EPICENTRO_KM',
    bins=[0, 70, 300, float('inf')],
    etiquetas=['Cercano', 'Intermedio', 'Lejano'],
    descripcion='Categorización de distancia al epicentro'
)
pipeline.agregar_transformador(categorizador)
print("\n" + "="*70)
print(f' Pipeline configurado con {len(pipeline.transformadores)} transformadores')
print("="*70)
print(f' Categorías registradas: {categorizador.get_categorias_registradas()}')

NameError: name 'PipelineProcesamiento' is not defined

### 5.3 Renombrado de Variables

In [ ]:
class RenombradoVariables(Transformer):
    """
    Renombra columnas según mapeo definido.
    """
    MAPEO_ESTANDAR = {
        'UTC Date': 'FECHA_UTC',
        'Profundidad': 'PROFUNDIDAD',
        'Magnitud': 'MAGNITUD',
        'Date': 'FECHA_LOCAL',
        'Hour': 'HORA_LOCAL',
        'Location': 'UBICACION',
        'Latitude': 'LATITUD',
        'Longitude': 'LONGITUD'
    }
    def __init__(self, mapeo: Optional[Dict[str, str]] = None):
        super().__init__("RenombradoVariables", "Renombra columnas")
        self.mapeo = mapeo if mapeo is not None else self.MAPEO_ESTANDAR
    def validar_entrada(self, df: pd.DataFrame) -> bool:
        return not df.empty
    def transformar(self, df: pd.DataFrame) -> pd.DataFrame:
        mapeo_aplicable = {k: v for k, v in self.mapeo.items() if k in df.columns}
        self._logger.info(f"Columnas renombradas: {len(mapeo_aplicable)}")
        return df.rename(columns=mapeo_aplicable)

In [ ]:
# PRUEBA 2: Renombrado de Variables
print("\n" + "="*100)
print("PRUEBA 2: RenombradoVariables")
print("="*100)
try:
    if 'df_cargado' in locals():
        renombrador = RenombradoVariables()
        df_renombrado = ejecutar_transformador_con_log(
            renombrador,
            df_cargado,
            nombre_prueba="RenombradoVariables - Estandariza nombres de columnas"
        )
        print(f"✓ Variables renombradas correctamente")
    else:
        print("⚠️  Advertencia: Ejecute primero la PRUEBA 1 (CargadorDatos)")
except Exception as e:
    print(f"✗ Error: {e}")
    import traceback
    traceback.print_exc()

## **6. Transformadores Avanzados**

### 6.1 Procesador de Fechas

In [ ]:
class ProcesadorFechaUTC(Transformer):
    """Descompone FECHA_UTC en año, mes, día"""
    def __init__(self):
        super().__init__("ProcesadorFechaUTC", "Extrae componentes de fecha")
    def validar_entrada(self, df: pd.DataFrame) -> bool:
        return 'FECHA_UTC' in df.columns
    def transformar(self, df: pd.DataFrame) -> pd.DataFrame:
        df_copia = df.copy()
        df_copia['FECHA_UTC'] = pd.to_datetime(df_copia['FECHA_UTC'], errors='coerce')
        df_copia['ANIO_UTC'] = df_copia['FECHA_UTC'].dt.year
        df_copia['MES_UTC'] = df_copia['FECHA_UTC'].dt.month
        df_copia['DIA_UTC'] = df_copia['FECHA_UTC'].dt.day
        self._logger.info("Fecha descompuesta en año/mes/día")
        return df_copia

In [ ]:
# PRUEBA 3: Procesador de Fecha UTC
print("\n" + "="*100)
print("PRUEBA 3: ProcesadorFechaUTC")
print("="*100)
try:
    if 'df_cargado' in locals():
        # Aplicar renombrado primero (prerequisito para las fechas)
        renombrador = RenombradoVariables()
        df_renombrado = renombrador.ejecutar(df_cargado)
        
        # Ahora aplicar procesador de fecha con logging detallado
        procesador = ProcesadorFechaUTC()
        df_con_fechas = ejecutar_transformador_con_log(
            procesador,
            df_renombrado,
            nombre_prueba="ProcesadorFechaUTC - Extrae ANIO_UTC, MES_UTC, DIA_UTC"
        )
        print(f"✓ Fechas extraídas exitosamente")
    else:
        print("⚠️  Advertencia: Ejecute primero la PRUEBA 1 (CargadorDatos)")
except Exception as e:
    print(f"✗ Error: {e}")
    import traceback
    traceback.print_exc()

### 6.2 Separador de Profundidad

### 6.4 Separador de Ubicación

In [ ]:
class SeparadorUbicacion(Transformer):
    """Separa UBICACION en distancia, dirección y nombre del lugar"""
    def __init__(self):
        super().__init__("SeparadorUbicacion", "Extrae distancia, dirección y lugar")
    def validar_entrada(self, df: pd.DataFrame) -> bool:
        return 'UBICACION' in df.columns
    def transformar(self, df: pd.DataFrame) -> pd.DataFrame:
        df_copia = df.copy()
        # Inicializar columnas
        df_copia['DISTANCIA_EPICENTRO_KM'] = pd.NA
        df_copia['DIRECCION'] = pd.NA
        df_copia['NOMBRE_LUGAR'] = pd.NA
        ubicacion_str = df_copia['UBICACION'].astype(str)
        # Patrón: "123 km al Norte de Santiago"
        patron_completo = r'(\d+)\s*km\s*al\s*([A-Za-z\s]+?)\s*de\s*(.+)'
        busqueda = ubicacion_str.str.extract(patron_completo, expand=True)
        mask_full = busqueda[0].notna()
        df_copia.loc[mask_full, 'DISTANCIA_EPICENTRO_KM'] = pd.to_numeric(busqueda.loc[mask_full, 0], errors='coerce')
        df_copia.loc[mask_full, 'DIRECCION'] = busqueda.loc[mask_full, 1].str.strip()
        df_copia.loc[mask_full, 'NOMBRE_LUGAR'] = busqueda.loc[mask_full, 2].str.strip()
        # Patrón alternativo: "123 km de Santiago"
        mask_remaining = df_copia['DISTANCIA_EPICENTRO_KM'].isna()
        patron_simple = r'(\d+)\s*km\s*de\s*(.+)'
        busqueda_simple = ubicacion_str[mask_remaining].str.extract(patron_simple, expand=True)
        mask_simple = busqueda_simple[0].notna()
        df_copia.loc[mask_remaining & mask_simple, 'DISTANCIA_EPICENTRO_KM'] = pd.to_numeric(busqueda_simple.loc[mask_simple, 0], errors='coerce')
        df_copia.loc[mask_remaining & mask_simple, 'DIRECCION'] = 'Directo'
        df_copia.loc[mask_remaining & mask_simple, 'NOMBRE_LUGAR'] = busqueda_simple.loc[mask_simple, 1].str.strip()
        # Texto libre para los restantes
        mask_final = df_copia['NOMBRE_LUGAR'].isna()
        df_copia.loc[mask_final, 'NOMBRE_LUGAR'] = ubicacion_str[mask_final].str.strip()
        self._logger.info("Ubicación separada en distancia/dirección/lugar")
        return df_copia

In [ ]:
# PRUEBA 6: Separador de Ubicación
print("\n" + "="*100)
print("PRUEBA 6: SeparadorUbicacion")
print("="*100)
try:
    if 'df_mag' in locals():
        separador_ub = SeparadorUbicacion()
        df_ub = ejecutar_transformador_con_log(
            separador_ub,
            df_mag,
            nombre_prueba="SeparadorUbicacion - Extrae DISTANCIA_EPICENTRO_KM, DIRECCION, NOMBRE_LUGAR"
        )
        print(f"✓ Ubicación separada exitosamente")
    else:
        print("⚠️  Advertencia: Ejecute pruebas anteriores primero")
except Exception as e:
    print(f"✗ Error: {e}")
    import traceback
    traceback.print_exc()

### 6.5 Estandarizador de Magnitud a MW

In [ ]:
class EstandarizadorMagnitudMW(Transformer):
    """
    Convierte diferentes escalas sísmicas a Mw (magnitud de momento).
    Fórmulas de conversión (Scordilis, 2006):
    - Ml, Mb, Mc: Mw = 0.85 * valor + 1.03
    - Ms: Mw = 0.67 * valor + 2.07
    """
    def __init__(self):
        super().__init__("EstandarizadorMagnitudMW", "Convierte escalas a Mw")
        self.conversiones = {
            'Ml': lambda x: 0.85 * x + 1.03,
            'Mlv': lambda x: 0.85 * x + 1.03,  # Mlv se trata como Ml
            'Mb': lambda x: 0.85 * x + 1.03,
            'Mc': lambda x: 0.85 * x + 1.03,
            'Ms': lambda x: 0.67 * x + 2.07
        }
    def validar_entrada(self, df: pd.DataFrame) -> bool:
        return 'MAGNITUD_VALOR' in df.columns and 'MAGNITUD_ESCALA' in df.columns
    def transformar(self, df: pd.DataFrame) -> pd.DataFrame:
        df_copia = df.copy()
        # Inicializar MAGNITUD_MW con valores originales
        df_copia['MAGNITUD_MW'] = df_copia['MAGNITUD_VALOR']
        # Recategorizar Mlv como Ml
        df_copia['MAGNITUD_ESCALA'] = df_copia['MAGNITUD_ESCALA'].astype(str)
        mlv_count = (df_copia['MAGNITUD_ESCALA'] == 'Mlv').sum()
        df_copia.loc[df_copia['MAGNITUD_ESCALA'] == 'Mlv', 'MAGNITUD_ESCALA'] = 'Ml'
        if mlv_count > 0:
            self._logger.info(f"Recategorizadas {mlv_count} magnitudes de 'Mlv' a 'Ml'")
        # Aplicar conversiones
        for escala, formula in self.conversiones.items():
            mask = df_copia['MAGNITUD_ESCALA'] == escala
            count = mask.sum()
            if count > 0:
                df_copia.loc[mask, 'MAGNITUD_MW'] = formula(df_copia.loc[mask, 'MAGNITUD_VALOR'])
                self._logger.info(f"Convertidas {count} magnitudes de '{escala}' a 'Mw'")
        return df_copia

In [ ]:
# PRUEBA 7: Estandarizador de Magnitud a MW
print("\n" + "="*100)
print("PRUEBA 7: EstandarizadorMagnitudMW")
print("="*100)
try:
    if 'df_ub' in locals():
        estandarizador = EstandarizadorMagnitudMW()
        df_mw = ejecutar_transformador_con_log(
            estandarizador,
            df_ub,
            nombre_prueba="EstandarizadorMagnitudMW - Convierte todas las escalas a Mw (Scordilis 2006)"
        )
        print(f"✓ Magnitudes estandarizadas a MW exitosamente")
    else:
        print("⚠️  Advertencia: Ejecute pruebas anteriores primero")
except Exception as e:
    print(f"✗ Error: {e}")
    import traceback
    traceback.print_exc()

### 6.6 Filtro Geográfico (Chile)

In [ ]:
class FiltroGeograficoChile(Transformer):
    """Filtra sismos que ocurrieron dentro del territorio chileno"""
    def __init__(self, lat_min=-56.0, lat_max=-17.0, lon_min=-90.0, lon_max=-66.0):
        super().__init__("FiltroGeograficoChile", "Filtra eventos en Chile")
        self.lat_min = lat_min
        self.lat_max = lat_max
        self.lon_min = lon_min
        self.lon_max = lon_max
        self._registros_eliminados = 0
    def validar_entrada(self, df: pd.DataFrame) -> bool:
        return 'LATITUD' in df.columns and 'LONGITUD' in df.columns
    def transformar(self, df: pd.DataFrame) -> pd.DataFrame:
        n_total = len(df)
        # Máscara para eventos DENTRO de Chile
        mask_chile = (
            (df['LATITUD'] >= self.lat_min) & (df['LATITUD'] <= self.lat_max) &
            (df['LONGITUD'] >= self.lon_min) & (df['LONGITUD'] <= self.lon_max)
        )
        df_filtrado = df[mask_chile].copy()
        self._registros_eliminados = n_total - len(df_filtrado)
        porcentaje = (self._registros_eliminados / n_total) * 100
        self._logger.info(f"Eliminados {self._registros_eliminados} registros fuera de Chile ({porcentaje:.2f}%)")
        self._logger.info(f"Registros conservados: {len(df_filtrado)}")
        return df_filtrado

In [ ]:
# PRUEBA 8: Filtro Geográfico Chile
print("\n" + "="*100)
print("PRUEBA 8: FiltroGeograficoChile")
print("="*100)
try:
    if 'df_mw' in locals():
        filtro = FiltroGeograficoChile()
        df_chile = ejecutar_transformador_con_log(
            filtro,
            df_mw,
            nombre_prueba="FiltroGeograficoChile - Mantiene solo eventos dentro de Chile"
        )
        print(f"✓ Filtro geográfico aplicado exitosamente")
        print(f"Rangos geográficos resultantes:")
        print(f"   Latitud: {df_chile['LATITUD'].min():.2f}° a {df_chile['LATITUD'].max():.2f}°")
        print(f"   Longitud: {df_chile['LONGITUD'].min():.2f}° a {df_chile['LONGITUD'].max():.2f}°")
    else:
        print("⚠️  Advertencia: Ejecute pruebas anteriores primero")
except Exception as e:
    print(f"✗ Error: {e}")
    import traceback
    traceback.print_exc()

### 6.7 Tratamiento de Valores Nulos

In [ ]:
class TratamientoNulos(Transformer):
    """
    Elimina filas con valores nulos.
    Estrategias:
    - 'eliminar': Elimina filas con cualquier valor nulo
    - 'columnas': Elimina solo filas con nulos en columnas especificadas
    """
    def __init__(self, estrategia='eliminar', columnas: Optional[List[str]] = None):
        super().__init__("TratamientoNulos", "Maneja valores nulos")
        self.estrategia = estrategia
        self.columnas = columnas
        self._nulos_eliminados = 0
    def validar_entrada(self, df: pd.DataFrame) -> bool:
        return not df.empty
    def transformar(self, df: pd.DataFrame) -> pd.DataFrame:
        filas_iniciales = len(df)
        if self.estrategia == 'eliminar':
            if self.columnas:
                df_limpio = df.dropna(subset=self.columnas).copy()
            else:
                df_limpio = df.dropna().copy()
        else:
            df_limpio = df.copy()
        self._nulos_eliminados = filas_iniciales - len(df_limpio)
        self._logger.info(f"Eliminadas {self._nulos_eliminados} filas con valores nulos")
        self._logger.info(f"Filas restantes: {len(df_limpio)}")
        return df_limpio

In [ ]:
# PRUEBA 9: Tratamiento de Nulos
print("\n" + "="*100)
print("PRUEBA 9: TratamientoNulos")
print("="*100)
try:
    if 'df_chile' in locals():
        tratador = TratamientoNulos(estrategia='eliminar')
        df_nulos = ejecutar_transformador_con_log(
            tratador,
            df_chile,
            nombre_prueba="TratamientoNulos - Elimina filas con valores nulos"
        )
        print(f"✓ Tratamiento de nulos completado")
        print(f"\nVerificación final:")
        print(f"   Total de valores nulos restantes: {df_nulos.isnull().sum().sum()}")
    else:
        print("⚠️  Advertencia: Ejecute pruebas anteriores primero")
except Exception as e:
    print(f"✗ Error: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
class SeparadorProfundidad(Transformer):
    """Separa valor numérico y unidad de profundidad"""
    def __init__(self):
        super().__init__("SeparadorProfundidad", "Extrae valor y unidad")
    def validar_entrada(self, df: pd.DataFrame) -> bool:
        return 'PROFUNDIDAD' in df.columns
    def transformar(self, df: pd.DataFrame) -> pd.DataFrame:
        df_copia = df.copy()
        patron = r'([0-9.]+)\s*([a-zA-Z]+)'
        extraido = df_copia['PROFUNDIDAD'].astype(str).str.extract(patron)
        df_copia['PROFUNDIDAD_VALOR'] = pd.to_numeric(extraido[0], errors='coerce')
        df_copia['PROFUNDIDAD_UNIDAD'] = extraido[1]
        self._logger.info("Profundidad separada en valor/unidad")
        return df_copia

In [ ]:
# PRUEBA 4: Separador de Profundidad
print("\n" + "="*100)
print("PRUEBA 4: SeparadorProfundidad")
print("="*100)
try:
    if 'df_con_fechas' in locals():
        separador_prof = SeparadorProfundidad()
        df_prof = ejecutar_transformador_con_log(
            separador_prof,
            df_con_fechas,
            nombre_prueba="SeparadorProfundidad - Extrae PROFUNDIDAD_VALOR y PROFUNDIDAD_UNIDAD"
        )
        print(f"✓ Profundidad separada exitosamente")
    else:
        print("⚠️  Advertencia: Ejecute pruebas anteriores primero")
except Exception as e:
    print(f"✗ Error: {e}")
    import traceback
    traceback.print_exc()

### 6.3 Separador de Magnitud

In [ ]:
class SeparadorMagnitud(Transformer):
    """Separa valor numérico y escala de magnitud"""
    def __init__(self):
        super().__init__("SeparadorMagnitud", "Extrae valor y escala")
    def validar_entrada(self, df: pd.DataFrame) -> bool:
        return 'MAGNITUD' in df.columns
    def transformar(self, df: pd.DataFrame) -> pd.DataFrame:
        df_copia = df.copy()
        patron = r'([0-9.]+)\s*([A-Za-z]+)'
        extraido = df_copia['MAGNITUD'].astype(str).str.extract(patron)
        df_copia['MAGNITUD_VALOR'] = pd.to_numeric(extraido[0], errors='coerce')
        df_copia['MAGNITUD_ESCALA'] = extraido[1]
        self._logger.info("Magnitud separada en valor/escala")
        return df_copia

In [ ]:
# PRUEBA 5: Separador de Magnitud
print("\n" + "="*100)
print("PRUEBA 5: SeparadorMagnitud")
print("="*100)
try:
    if 'df_prof' in locals():
        separador_mag = SeparadorMagnitud()
        df_mag = ejecutar_transformador_con_log(
            separador_mag,
            df_prof,
            nombre_prueba="SeparadorMagnitud - Extrae MAGNITUD_VALOR y MAGNITUD_ESCALA"
        )
        print(f"✓ Magnitud separada exitosamente")
        print(f"\nDistribución de escalas:")
        print(df_mag['MAGNITUD_ESCALA'].value_counts())
    else:
        print("⚠️  Advertencia: Ejecute pruebas anteriores primero")
except Exception as e:
    print(f"✗ Error: {e}")
    import traceback
    traceback.print_exc()

## **7. Pipeline de Procesamiento (Builder Pattern)**

Orquestador que construye y ejecuta pipelines de transformación con API fluida.

In [ ]:
class PipelineProcesamiento:
    """
    Builder Pattern: Construye pipeline de transformaciones.
    Características:
    - API fluida (fluent interface)
    - Composición de transformadores
    - Tracking de resultados
    - Logging comprehensivo
    """
    def __init__(self):
        self.transformadores: List[Transformer] = []
        self.df_datos = None
        self.resultados: List[ResultadoTransformacion] = []
        self._logger = logging.getLogger("Pipeline")
    def cargar_datos(self, ruta: str, encoding: str = 'utf-8'):
        """Carga datos iniciales"""
        cargador = CargadorDatos(ruta, encoding)
        self.df_datos = cargador.ejecutar(pd.DataFrame())
        self._logger.info(f"Datos cargados: {self.df_datos.shape}")
        return self
    def agregar_transformador(self, transformador: Transformer):
        """Agrega transformador al pipeline"""
        self.transformadores.append(transformador)
        self._logger.info(f"Transformador agregado: {transformador.nombre}")
        return self
    def ejecutar(self) -> pd.DataFrame:
        """
        Ejecuta todos los transformadores secuencialmente.
        Registra resultados de cada transformación.
        """
        if self.df_datos is None:
            raise ValueError("No hay datos cargados. Usar cargar_datos() primero")
        df = self.df_datos.copy()
        self._logger.info(f"Iniciando pipeline con {len(self.transformadores)} transformadores")
        for i, transformador in enumerate(self.transformadores, 1):
            filas_antes = len(df)
            cols_antes = len(df.columns)
            df = transformador.ejecutar(df)
            resultado = ResultadoTransformacion(
                transformador=transformador.nombre,
                estado="OK",
                filas_entrada=filas_antes,
                filas_salida=len(df),
                columnas_entrada=cols_antes,
                columnas_salida=len(df.columns),
                tiempo_ejecucion=transformador.get_tiempo_ejecucion() or 0.0
            )
            self.resultados.append(resultado)
            self._logger.info(f"[{i}/{len(self.transformadores)}] {transformador.nombre} completado")
        self._logger.info("Pipeline ejecutado exitosamente")
        return df
    def get_resumen(self) -> str:
        """Genera resumen de ejecución"""
        if not self.resultados:
            return "Pipeline no ejecutado aún"
        tiempo_total = sum(r.tiempo_ejecucion for r in self.resultados)
        resumen = f"Pipeline ejecutado: {len(self.resultados)} transformadores\n"
        resumen += f"Tiempo total: {tiempo_total:.4f}s\n\n"
        for res in self.resultados:
            resumen += f"{res}\n\n"
        return resumen
    def __repr__(self) -> str:
        return f"PipelineProcesamiento({len(self.transformadores)} transformadores)"

In [ ]:
# PRUEBA 11: PipelineProcesamiento (Builder Pattern)
print("\n" + "="*100)
print("PRUEBA 11: PipelineProcesamiento (Builder Pattern)")
print("="*100)
try:
    # Crear instancia del pipeline
    pipeline_prueba = PipelineProcesamiento()
    
    # 1. Cargar datos
    print("📥 Paso 1: Cargando datos...")
    pipeline_prueba.cargar_datos(RUTA_DATASET)
    print(f"   ✓ {pipeline_prueba.df_datos.shape[0]:,} filas cargadas")
    
    # 2. Agregar transformadores
    print("\n  Paso 2: Agregando transformadores al pipeline...")
    transformadores = [
        ("Limpieza de duplicados", LimpiezaDuplicados()),
        ("Renombrado de variables", RenombradoVariables()),
        ("Extracción de fechas", ProcesadorFechaUTC()),
        ("Separación de profundidad", SeparadorProfundidad()),
        ("Separación de magnitud", SeparadorMagnitud()),
        ("Separación de ubicación", SeparadorUbicacion()),
        ("Estandarización de magnitud Mw", EstandarizadorMagnitudMW()),
        ("Filtro geográfico Chile", FiltroGeograficoChile()),
        ("Tratamiento de nulos", TratamientoNulos()),
    ]
    
    for nombre, transformador in transformadores:
        pipeline_prueba.agregar_transformador(transformador)
        print(f"   ✓ Agregado: {nombre}")
    
    # 3. Ejecutar pipeline
    print("\n  Paso 3: Ejecutando pipeline completo...")
    df_resultado = pipeline_prueba.ejecutar()
    
    print(f"\n    Pipeline ejecutado exitosamente")
    print(f"\n📊 RESUMEN DE RESULTADOS:")
    print(f"   Registros iniciales: {pipeline_prueba.df_datos.shape[0]:,}")
    print(f"   Registros finales: {len(df_resultado):,}")
    print(f"   Columnas generadas: {len(df_resultado.columns)}")
    print(f"   Transformadores ejecutados: {len(pipeline_prueba.resultados)}")
    
    print(f"\n📈 DETALLES DE TRANSFORMACIONES:")
    for i, resultado in enumerate(pipeline_prueba.resultados, 1):
        print(f"   {i}. {resultado.transformador}")
        print(f"      Estado: {resultado.estado}")
        print(f"      Filas: {resultado.filas_entrada:,} → {resultado.filas_salida:,}")
        print(f"      Tiempo: {resultado.tiempo_ejecucion:.4f}s")
    
except Exception as e:
    print(f"✗ Error crítico: {e}")
    import traceback
    traceback.print_exc()

## 🎯 Resumen: Cómo usar las Pruebas con Logging Detallado

Cada prueba ahora utiliza la función `ejecutar_transformador_con_log()` que **muestra cada etapa del proceso**:

### ✅ Lo que ves en cada ejecución:

1. **📥 ENTRADA** - Información del DataFrame que entra
2. **✓ VALIDANDO** - Resultado de la validación de precondiciones
3. **⚙️ TRANSFORMANDO** - Ejecución y tiempo de proceso
4. **📊 CAMBIOS REALIZADOS** - Qué cambió exactamente
   - Filas añadidas/eliminadas
   - Columnas nuevas
   - Columnas eliminadas
5. **📋 MUESTREO DE DATOS** - Primeras 3 filas con cambios
6. **📈 ESTADÍSTICAS** - Memoria, nulos, etc.

### 🚀 Ejecución paso a paso:

```python
# Las pruebas deben ejecutarse EN ORDEN:
1. Ejecutar: PRUEBA 1 - CargadorDatos
2. Ejecutar: PRUEBA 2 - RenombradoVariables  
3. Ejecutar: PRUEBA 3 - ProcesadorFechaUTC
4. Ejecutar: PRUEBA 4 - SeparadorProfundidad
5. Ejecutar: PRUEBA 5 - SeparadorMagnitud
6. Ejecutar: PRUEBA 6 - SeparadorUbicacion
7. Ejecutar: PRUEBA 7 - EstandarizadorMagnitudMW
8. Ejecutar: PRUEBA 8 - FiltroGeograficoChile
9. Ejecutar: PRUEBA 9 - TratamientoNulos
10. Ejecutar: PRUEBA 10 - CategorizadorDatos
11. Ejecutar: PRUEBA 11 - PipelineProcesamiento
```

### ✨ Ventajas de este enfoque:

- ✅ **Validación visible** - Ves si las precondiciones se cumplen
- ✅ **Transformaciones claras** - Qué cambió en cada paso
- ✅ **Debugging fácil** - Si algo falla, sabes exactamente dónde
- ✅ **Evidencia por pantalla** - Sin necesidad de debuggers externos
- ✅ **Flujo transparente** - Entiendes la lógica de la clase Transformer base

## **8. Categorizador de Datos (Strategy Pattern)**

Permite registrar estrategias de categorización dinámicamente.

In [ ]:
class CategorizadorDatos(Transformer):
    """
    Strategy Pattern: Categoriza variables numéricas.
    Permite registrar múltiples estrategias de categorización
    que se aplican dinámicamente.
    """
    def __init__(self):
        super().__init__("Categorizador", "Aplica categorizaciones")
        self.categorias: Dict[str, ConfiguracionCategoria] = {}
    def registrar_categoria(self, nombre: str, columna: str, 
                          bins: List[float], etiquetas: List[str],
                          descripcion: str = ""):
        """
        Registra una nueva estrategia de categorización.
        Args:
            nombre: Nombre de la nueva columna categórica
            columna: Columna numérica fuente
            bins: Rangos para categorización
            etiquetas: Etiquetas de categorías
            descripcion: Descripción opcional
        """
        config = ConfiguracionCategoria(columna, bins, etiquetas, descripcion)
        if not config.validar():
            raise ValueError(f"Configuración inválida para {nombre}")
        self.categorias[nombre] = config
        self._logger.info(f"Categoría registrada: {nombre}")
        return self
    def validar_entrada(self, df: pd.DataFrame) -> bool:
        columnas_req = {c.columna for c in self.categorias.values()}
        return columnas_req.issubset(set(df.columns))
    def transformar(self, df: pd.DataFrame) -> pd.DataFrame:
        df_copia = df.copy()
        for nombre, config in self.categorias.items():
            df_copia[nombre] = pd.cut(
                df_copia[config.columna],
                bins=config.bins,
                labels=config.etiquetas,
                include_lowest=True
            )
            self._logger.info(f"Categoría creada: {nombre}")
        return df_copia
    def get_categorias_registradas(self) -> List[str]:
        return list(self.categorias.keys())

In [ ]:
# PRUEBA 10: CategorizadorDatos (Strategy Pattern)
print("\n" + "="*100)
print("PRUEBA 10: CategorizadorDatos (Strategy Pattern)")
print("="*100)
try:
    if 'df_nulos' in locals():
        categorizador = CategorizadorDatos()
        
        # Registrar categorías
        print("📝 Registrando estrategias de categorización...")
        categorizador.registrar_categoria(
            'PROF_CAT',
            'PROFUNDIDAD_VALOR',
            [0, 70, 300, float('inf')],
            ['somero', 'intermedio', 'profundo'],
            'Clasificación de profundidad sísmica'
        )
        print("   ✓ Categoría: PROF_CAT (profundidad sísmica)")
        
        categorizador.registrar_categoria(
            'MAG_CAT',
            'MAGNITUD_MW',
            [0, 2, 4, 5, 6, 7, 8, float('inf')],
            ['micro', 'menor', 'ligero', 'moderado', 'fuerte', 'mayor', 'grande'],
            'Clasificación de magnitud Mw'
        )
        print("   ✓ Categoría: MAG_CAT (magnitud Mw)")
        
        print(f"\nCategorías registradas: {categorizador.get_categorias_registradas()}")
        
        # Aplicar categorización
        df_categorizado = ejecutar_transformador_con_log(
            categorizador,
            df_nulos,
            nombre_prueba="CategorizadorDatos (Strategy Pattern) - Aplica categorización"
        )
        
        print(f"✓ Categorización aplicada exitosamente")
        print(f"\n📊 Distribuciones generadas:")
        print(f"\n   PROF_CAT:")
        print(df_categorizado['PROF_CAT'].value_counts().sort_index())
        print(f"\n   MAG_CAT:")
        print(df_categorizado['MAG_CAT'].value_counts())
        
    else:
        print("⚠️  Advertencia: Ejecute pruebas anteriores primero")
except Exception as e:
    print(f"✗ Error: {e}")
    import traceback
    traceback.print_exc()

## **9. Construcción del Pipeline Completo**

Configuración y construcción del pipeline con todos los transformadores.

In [ ]:
# Ruta del dataset
RUTA_DATASET = r'../../data/raw/EarthquakesChile_2000-2024.csv'
# Construcción del pipeline usando Builder Pattern
pipeline = PipelineProcesamiento()
# API Fluida: Cargar datos
pipeline.cargar_datos(RUTA_DATASET)
# Agregar transformadores básicos
pipeline.agregar_transformador(LimpiezaDuplicados())
pipeline.agregar_transformador(RenombradoVariables())
# Agregar transformadores avanzados
pipeline.agregar_transformador(ProcesadorFechaUTC())
pipeline.agregar_transformador(SeparadorProfundidad())
pipeline.agregar_transformador(SeparadorMagnitud())
# Configurar categorizador con Strategy Pattern
categorizador = CategorizadorDatos()
categorizador.registrar_categoria(
    nombre='PROF_CAT',
    columna='PROFUNDIDAD_VALOR',
    bins=[0, 70, 300, float('inf')],
    etiquetas=['somero', 'intermedio', 'profundo'],
    descripcion='Categorización de profundidad sísmica'
)
categorizador.registrar_categoria(
    nombre='MAG_CAT',
    columna='MAGNITUD_VALOR',
    bins=[0, 2, 4, 5, 6, 7, 8, float('inf')],
    etiquetas=['micro', 'menor', 'ligero', 'moderado', 'fuerte', 'mayor', 'grande'],
    descripcion='Categorización de magnitud sísmica'
)
pipeline.agregar_transformador(categorizador)
print(f' Pipeline configurado con {len(pipeline.transformadores)} transformadores')
print(f' Categorías registradas: {categorizador.get_categorias_registradas()}')

## **10. Ejecución del Pipeline**

Ejecutamos el pipeline completo y mostramos resultados.

In [ ]:
# Ejecutar pipeline
try:
    df_final = pipeline.ejecutar()
    print('='*70)
    print('='*70)
    print(f'\nFilas finales: {df_final.shape[0]:,}')
    print(f'Columnas finales: {df_final.shape[1]}')
    print(f'\nColumnas disponibles:')
    print(df_final.columns.tolist())
except FileNotFoundError:
    print(f' Error: Archivo no encontrado en {RUTA_DATASET}')
    print('Verifica que el archivo exista en la ruta correcta.')
except Exception as e:
    print(f' Error durante la ejecución: {str(e)}')

## **11. Inspección de Resultados**

In [ ]:
# Mostrar primeras filas
if 'df_final' in locals():
    print('Primeras 10 filas del dataset procesado:\n')
    print(df_final.head(10))
    print('\n' + '='*70)
    print('Información del DataFrame:')
    print('='*70)
    df_final.info()
    print('\n' + '='*70)
    print('Estadísticas descriptivas:')
    print('='*70)
    print(df_final.describe())

## **12. Resumen de Transformaciones**

In [ ]:
# Mostrar resumen del pipeline
if 'pipeline' in locals():
    print(pipeline.get_resumen())

## **13. Visualizaciones**

Análisis visual de los datos procesados.

## **11. Pipeline Completo - Ejecución Rápida**

Esta sección permite ejecutar todo el pipeline de procesamiento con una sola llamada, facilitando las pruebas y validaciones del sistema completo.

In [ ]:
def ejecutar_pipeline_completo(ruta_dataset: str, verbose: bool = True) -> pd.DataFrame:
    """
    Ejecuta el pipeline completo de procesamiento de datos sísmicos.
    Esta función orquesta todas las etapas del procesamiento:
    1. Carga de datos
    2. Limpieza de duplicados
    3. Renombrado de variables
    4. Procesamiento de fecha UTC
    5. Separación de profundidad
    6. Separación de magnitud
    7. Separación de ubicación
    8. Estandarización de magnitud a Mw
    9. Filtro geográfico (Chile)
    10. Tratamiento de nulos
    11. Categorización de datos
    Parámetros
    ----------
    ruta_dataset : str
        Ruta al archivo CSV con los datos sísmicos
    verbose : bool, optional
        Si True, muestra información detallada del proceso (default: True)
    Retorna
    -------
    pd.DataFrame
        DataFrame procesado y listo para análisis
    Ejemplo
    -------
    >>> df_procesado = ejecutar_pipeline_completo('../../data/raw/EarthquakesChile_2000-2024.csv')
    """
    import time
    inicio_total = time.time()
    if verbose:
        print("" + "" * 78 + "")
        print("" + "" * 78 + "")
        print()
    try:
        # 1. Construcción del pipeline
        if verbose:
        pipeline = PipelineProcesamiento()
        # 2. Carga de datos
        pipeline.cargar_datos(ruta_dataset)
        if verbose:
            print(" Datos cargados correctamente\n")
        # 3. Agregar transformadores en el orden correcto
        transformadores = [
            ("Limpieza de duplicados", LimpiezaDuplicados()),
            ("Renombrado de variables", RenombradoVariables()),
            ("Procesamiento de fecha UTC", ProcesadorFechaUTC()),
            ("Separación de profundidad", SeparadorProfundidad()),
            ("Separación de magnitud", SeparadorMagnitud()),
            ("Separación de ubicación", SeparadorUbicacion()),
            ("Estandarización a Mw", EstandarizadorMagnitudMW()),
            ("Filtro geográfico", FiltroGeograficoChile()),
            ("Tratamiento de nulos", TratamientoNulos())
        ]
        for nombre, transformador in transformadores:
            pipeline.agregar_transformador(transformador)
            if verbose:
                print(f" Agregado: {nombre}")
        # 4. Configurar categorización
        categorizador = CategorizadorDatos()
        # Categorías de profundidad
        categorizador.registrar_estrategia(
            'PROFUNDIDAD_VALOR',
            'CATEGORIA_PROFUNDIDAD',
            ConfiguracionCategoria(
                bins=[0, 70, 300, float('inf')],
                labels=['somero', 'intermedio', 'profundo'],
                descripcion='Clasificación de profundidad sísmica'
            )
        )
        # Categorías de magnitud
        categorizador.registrar_estrategia(
            'MAGNITUD_MW',
            'CATEGORIA_MAGNITUD',
            ConfiguracionCategoria(
                bins=[0, 2.0, 4.0, 5.0, 6.0, 7.0, 8.0, float('inf')],
                labels=['Micro', 'Menor', 'Ligero', 'Moderado', 'Fuerte', 'Mayor', 'Grande'],
                descripcion='Clasificación de magnitud Mw'
            )
        )
        # Categorías de distancia
        categorizador.registrar_estrategia(
            'DISTANCIA_EPICENTRO_KM',
            'CATEGORIA_DISTANCIA',
            ConfiguracionCategoria(
                bins=[0, 70, 300, float('inf')],
                labels=['Cercano', 'Intermedio', 'Lejano'],
                descripcion='Clasificación de distancia al epicentro'
            )
        )
        pipeline.agregar_transformador(categorizador)
        if verbose:
            print()
        # 5. Ejecutar pipeline
        if verbose:
            print("" * 80)
        df_procesado = pipeline.ejecutar()
        tiempo_total = time.time() - inicio_total
        # 6. Resumen final
        if verbose:
            print("" * 80)
            print(f"  Tiempo total de ejecución: {tiempo_total:.2f} segundos")
            print(f" Registros procesados: {len(df_procesado):,}")
            print(f" Columnas generadas: {len(df_procesado.columns)}")
            print(f" Memoria utilizada: {df_procesado.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
            print()
            print(" Columnas disponibles:")
            for i, col in enumerate(df_procesado.columns, 1):
                print(f"   {i:2d}. {col}")
            print()
        return df_procesado
    except Exception as e:
        if verbose:
            print(f"\n ERROR en el pipeline: {str(e)}")
            import traceback
            traceback.print_exc()
        raise
# Información de uso
print("" + "" * 78 + "")
print("" + "" * 78 + "")
print("\n Uso:")
print("\n Parámetros:")
print("   - ruta_dataset: Ruta al archivo CSV")
print("   - verbose: True para ver detalles (default), False para modo silencioso")
print("\n Esta función ejecuta automáticamente todas las etapas del procesamiento POO")
print()

### **11.1 Ejecución del Pipeline**

Ejecuta el pipeline completo para procesar los datos sísmicos.

In [ ]:
# Ejecutar el pipeline completo
df_final = ejecutar_pipeline_completo(RUTA_DATASET)
# Mostrar resultados
print("\n" + "=" * 80)
print("VISTA PREVIA DE LOS DATOS PROCESADOS")
print("=" * 80)
print("\n Primeras 10 filas:")
display(df_final.head(10))
print("\n Últimas 5 filas:")
display(df_final.tail())
print("\n Información del DataFrame:")
df_final.info()
print("\n Estadísticas descriptivas:")
display(df_final.describe())

### **11.2 Análisis de Categorías Generadas**

Visualización de las distribuciones de las categorías creadas por el pipeline.

In [ ]:
# Análisis de las categorías generadas
print("" + "" * 78 + "")
print("" + "" * 78 + "\n")
# 1. Categoría de Profundidad
if 'CATEGORIA_PROFUNDIDAD' in df_final.columns:
    print("\n CATEGORÍA DE PROFUNDIDAD")
    print("" * 80)
    conteo_prof = df_final['CATEGORIA_PROFUNDIDAD'].value_counts()
    porcentaje_prof = (conteo_prof / len(df_final) * 100).round(2)
    for cat, count in conteo_prof.items():
        pct = porcentaje_prof[cat]
        barra = "█" * int(pct / 2)
        print(f"  {cat:12s}: {count:6,} ({pct:5.2f}%) {barra}")
# 2. Categoría de Magnitud
if 'CATEGORIA_MAGNITUD' in df_final.columns:
    print("\n\n CATEGORÍA DE MAGNITUD")
    print("" * 80)
    conteo_mag = df_final['CATEGORIA_MAGNITUD'].value_counts()
    porcentaje_mag = (conteo_mag / len(df_final) * 100).round(2)
    for cat, count in conteo_mag.items():
        pct = porcentaje_mag[cat]
        barra = "█" * int(pct / 2)
        print(f"  {cat:12s}: {count:6,} ({pct:5.2f}%) {barra}")
# 3. Categoría de Distancia
if 'CATEGORIA_DISTANCIA' in df_final.columns:
    print("\n\n CATEGORÍA DE DISTANCIA AL EPICENTRO")
    print("" * 80)
    conteo_dist = df_final['CATEGORIA_DISTANCIA'].value_counts()
    porcentaje_dist = (conteo_dist / len(df_final) * 100).round(2)
    for cat, count in conteo_dist.items():
        pct = porcentaje_dist[cat]
        barra = "█" * int(pct / 2)
        print(f"  {cat:12s}: {count:6,} ({pct:5.2f}%) {barra}")
print("\n\n Todas las categorías han sido generadas correctamente")
print(f" Total de registros categorizados: {len(df_final):,}")

### **11.3 Validación y Exportación (Opcional)**

Validaciones finales del dataset procesado y opción para exportar.

In [ ]:
# Validaciones finales del dataset
print("" + "" * 78 + "")
print("" + " " * 28 + "VALIDACIONES FINALES" + " " * 30 + "")
print("" + "" * 78 + "\n")
# 1. Verificar que no hay valores nulos
nulos_por_columna = df_final.isnull().sum()
total_nulos = nulos_por_columna.sum()
print(" Verificación de valores nulos:")
if total_nulos == 0:
    print("   No hay valores nulos en el dataset")
else:
    print(f"    Se encontraron {total_nulos} valores nulos:")
    print(nulos_por_columna[nulos_por_columna > 0])
# 2. Verificar que no hay duplicados
duplicados = df_final.duplicated().sum()
print(f"\n Verificación de duplicados:")
if duplicados == 0:
    print("   No hay registros duplicados")
else:
    print(f"    Se encontraron {duplicados} registros duplicados")
# 3. Verificar rangos de datos
print("\n Verificación de rangos de datos:")
print(f"  • Profundidad: [{df_final['PROFUNDIDAD_VALOR'].min():.1f}, {df_final['PROFUNDIDAD_VALOR'].max():.1f}] km")
print(f"  • Magnitud MW: [{df_final['MAGNITUD_MW'].min():.2f}, {df_final['MAGNITUD_MW'].max():.2f}]")
print(f"  • Latitud: [{df_final['LATITUD'].min():.2f}, {df_final['LATITUD'].max():.2f}]°")
print(f"  • Longitud: [{df_final['LONGITUD'].min():.2f}, {df_final['LONGITUD'].max():.2f}]°")
# 4. Verificar que todas las categorías existen
print("\n Verificación de columnas de categorización:")
columnas_categorias = ['CATEGORIA_PROFUNDIDAD', 'CATEGORIA_MAGNITUD', 'CATEGORIA_DISTANCIA']
for col in columnas_categorias:
    if col in df_final.columns:
        print(f"   {col}: {df_final[col].nunique()} categorías únicas")
    else:
        print(f"   {col}: NO ENCONTRADA")
print("\n\n" + "" * 80)
print("" * 80)
# Opción para exportar (comentada por defecto)
# Descomenta las siguientes líneas si deseas exportar el dataset procesado
"""
print("\n Exportando dataset procesado...")
ruta_exportacion = '../../data/processed/terremotos_chile_procesado.csv'
df_final.to_csv(ruta_exportacion, index=False, encoding='utf-8')
print(f" Dataset exportado exitosamente a: {ruta_exportacion}")
"""
print("\n Tip: Para exportar el dataset, descomenta el bloque de código de exportación.")

In [ ]:
if 'df_final' in locals():
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    # 1. Distribución de Magnitudes
    if 'MAGNITUD_VALOR' in df_final.columns:
        axes[0, 0].hist(df_final['MAGNITUD_VALOR'].dropna(), bins=50, 
                        edgecolor='black', alpha=0.7, color='steelblue')
        axes[0, 0].set_title('Distribución de Magnitudes', fontsize=14, fontweight='bold')
        axes[0, 0].set_xlabel('Magnitud')
        axes[0, 0].set_ylabel('Frecuencia')
        axes[0, 0].grid(True, alpha=0.3)
    # 2. Distribución de Profundidades
    if 'PROFUNDIDAD_VALOR' in df_final.columns:
        axes[0, 1].hist(df_final['PROFUNDIDAD_VALOR'].dropna(), bins=50,
                        edgecolor='black', alpha=0.7, color='coral')
        axes[0, 1].set_title('Distribución de Profundidades', fontsize=14, fontweight='bold')
        axes[0, 1].set_xlabel('Profundidad (km)')
        axes[0, 1].set_ylabel('Frecuencia')
        axes[0, 1].grid(True, alpha=0.3)
    # 3. Terremotos por Año
    if 'ANIO_UTC' in df_final.columns:
        terremotos_anio = df_final['ANIO_UTC'].value_counts().sort_index()
        axes[1, 0].plot(terremotos_anio.index, terremotos_anio.values,
                        marker='o', linewidth=2, color='green', markersize=6)
        axes[1, 0].set_title('Terremotos por Año', fontsize=14, fontweight='bold')
        axes[1, 0].set_xlabel('Año')
        axes[1, 0].set_ylabel('Cantidad de Terremotos')
        axes[1, 0].grid(True, alpha=0.3)
    # 4. Magnitud vs Profundidad
    if 'MAGNITUD_VALOR' in df_final.columns and 'PROFUNDIDAD_VALOR' in df_final.columns:
        scatter = axes[1, 1].scatter(
            df_final['PROFUNDIDAD_VALOR'],
            df_final['MAGNITUD_VALOR'],
            alpha=0.5, s=30,
            c=df_final['MAGNITUD_VALOR'],
            cmap='viridis'
        )
        axes[1, 1].set_title('Magnitud vs Profundidad', fontsize=14, fontweight='bold')
        axes[1, 1].set_xlabel('Profundidad (km)')
        axes[1, 1].set_ylabel('Magnitud')
        axes[1, 1].grid(True, alpha=0.3)
        plt.colorbar(scatter, ax=axes[1, 1], label='Magnitud')
    plt.tight_layout()
    plt.show()

## **14. Análisis por Categorías**

In [ ]:
if 'df_final' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    # Distribución de categorías de magnitud
    if 'MAG_CAT' in df_final.columns:
        mag_counts = df_final['MAG_CAT'].value_counts().sort_index()
        axes[0].bar(range(len(mag_counts)), mag_counts.values, 
                   tick_label=mag_counts.index, color='skyblue', edgecolor='black')
        axes[0].set_title('Distribución por Categoría de Magnitud', fontsize=12, fontweight='bold')
        axes[0].set_xlabel('Categoría')
        axes[0].set_ylabel('Frecuencia')
        axes[0].tick_params(axis='x', rotation=45)
        axes[0].grid(axis='y', alpha=0.3)
    # Distribución de categorías de profundidad
    if 'PROF_CAT' in df_final.columns:
        prof_counts = df_final['PROF_CAT'].value_counts().sort_index()
        axes[1].bar(range(len(prof_counts)), prof_counts.values,
                   tick_label=prof_counts.index, color='lightcoral', edgecolor='black')
        axes[1].set_title('Distribución por Categoría de Profundidad', fontsize=12, fontweight='bold')
        axes[1].set_xlabel('Categoría')
        axes[1].set_ylabel('Frecuencia')
        axes[1].tick_params(axis='x', rotation=45)
        axes[1].grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

## **15. Validación Técnica y Verificación**

Comprobaciones de calidad de datos.

In [ ]:
if 'df_final' in locals():
    print('='*70)
    print('='*70)
    # Valores nulos
    print('\n1. VALORES NULOS POR COLUMNA:')
    nulos = df_final.isnull().sum()
    nulos_pct = (nulos / len(df_final)) * 100
    for col in nulos[nulos > 0].index:
        print(f'   {col}: {nulos[col]} ({nulos_pct[col]:.2f}%)')
    if nulos.sum() == 0:
        print('    No se encontraron valores nulos')
    # Duplicados
    print('\n2. DUPLICADOS:')
    duplicados = df_final.duplicated().sum()
    print(f'   Total de duplicados: {duplicados}')
    if duplicados == 0:
        print('    No se encontraron duplicados')
    # Tipos de datos
    print('\n3. TIPOS DE DATOS:')
    for col in df_final.columns:
        print(f'   {col}: {df_final[col].dtype}')
    # Rangos de valores
    print('\n4. RANGOS DE VALORES:')
    if 'MAGNITUD_VALOR' in df_final.columns:
        print(f'   Magnitud: [{df_final["MAGNITUD_VALOR"].min():.2f}, {df_final["MAGNITUD_VALOR"].max():.2f}]')
    if 'PROFUNDIDAD_VALOR' in df_final.columns:
        print(f'   Profundidad: [{df_final["PROFUNDIDAD_VALOR"].min():.2f}, {df_final["PROFUNDIDAD_VALOR"].max():.2f}] km')
    if 'ANIO_UTC' in df_final.columns:
        print(f'   Rango temporal: [{int(df_final["ANIO_UTC"].min())}, {int(df_final["ANIO_UTC"].max())}]')

## **16. Análisis de Eficiencia y Optimización**

Mediciones de performance del pipeline.

In [ ]:
if 'pipeline' in locals() and pipeline.resultados:
    print('='*70)
    print('='*70)
    # Tiempo por transformador
    tiempos = [(r.transformador, r.tiempo_ejecucion) for r in pipeline.resultados]
    tiempo_total = sum(t[1] for t in tiempos)
    print('\nTIEMPO POR TRANSFORMADOR:')
    for nombre, tiempo in tiempos:
        porcentaje = (tiempo / tiempo_total) * 100
        print(f'   {nombre:30s}: {tiempo:.4f}s ({porcentaje:.1f}%)')
    print(f'\nTIEMPO TOTAL: {tiempo_total:.4f}s')
    # Gráfico de tiempos
    fig, ax = plt.subplots(figsize=(12, 6))
    nombres = [t[0] for t in tiempos]
    tiempos_val = [t[1] for t in tiempos]
    bars = ax.barh(nombres, tiempos_val, color='teal', edgecolor='black')
    ax.set_xlabel('Tiempo (segundos)')
    ax.set_title('Tiempo de Ejecución por Transformador', fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    # Agregar valores en las barras
    for i, (bar, tiempo) in enumerate(zip(bars, tiempos_val)):
        ax.text(tiempo, i, f' {tiempo:.4f}s', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()

## **17. Conclusiones**

Este notebook implementa un sistema completo de análisis sísmico con arquitectura POO profesional, cumpliendo todos los requisitos:

### Logros Técnicos:
1. **Arquitectura Modular**: Sistema extensible y mantenible
2. **Patrones de Diseño**: 5 patrones implementados correctamente
3. **Principios SOLID**: Aplicados en toda la arquitectura
4. **Validación Completa**: Verificación rigurosa en cada etapa
5. **Documentación Exhaustiva**: Código autodocumentado
6. **Optimización**: Mediciones y análisis de eficiencia
7. **Trazabilidad**: Sistema de logging comprehensivo

### Dataset Procesado:
- Limpieza de duplicados ✓
- Renombrado de variables ✓
- Extracción de componentes temporales ✓
- Separación de valores compuestos ✓
- Categorización de variables ✓
- Validación de integridad ✓

### Visualizaciones Generadas:
- Distribuciones de magnitud y profundidad ✓
- Series temporales de actividad sísmica ✓
- Relaciones entre variables ✓
- Análisis categórico ✓

**El sistema está listo para análisis avanzados y extensiones futuras.**

# Analisis de Terremotos Chile (2000-2024) - F3 POO

In [ ]:
import pandas as pd
import numpy as np
from abc import ABC, abstractmethod
from typing import List

## Clase Base Transformer

In [ ]:
class Transformer(ABC):
    def __init__(self, nombre: str):
        self.nombre = nombre
    @abstractmethod
    def validar_entrada(self, df: pd.DataFrame) -> bool:
        pass
    @abstractmethod
    def transformar(self, df: pd.DataFrame) -> pd.DataFrame:
        pass
    def ejecutar(self, df: pd.DataFrame) -> pd.DataFrame:
        if not self.validar_entrada(df):
            raise ValueError('Entrada invalida')
        return self.transformar(df)

## **12.5 Análisis de Categorías**

Análisis detallado de las categorizaciones aplicadas.

In [ ]:
if 'df_final' in locals():
    print('='*70)
    print('='*70)
    # Categorías de profundidad
    if 'CATEGORIA_PROFUNDIDAD' in df_final.columns:
        print('\n1. DISTRIBUCIÓN POR PROFUNDIDAD:')
        prof_dist = df_final['CATEGORIA_PROFUNDIDAD'].value_counts().sort_index()
        for cat, count in prof_dist.items():
            pct = (count / len(df_final)) * 100
            print(f'   {cat:15s}: {count:6,} ({pct:5.2f}%)')
    # Categorías de magnitud
    if 'CATEGORIA_MAGNITUD' in df_final.columns:
        print('\n2. DISTRIBUCIÓN POR MAGNITUD:')
        mag_dist = df_final['CATEGORIA_MAGNITUD'].value_counts().sort_index()
        for cat, count in mag_dist.items():
            pct = (count / len(df_final)) * 100
            print(f'   {cat:15s}: {count:6,} ({pct:5.2f}%)')
    # Categorías de distancia
    if 'CATEGORIA_DISTANCIA' in df_final.columns:
        print('\n3. DISTRIBUCIÓN POR DISTANCIA AL EPICENTRO:')
        dist_dist = df_final['CATEGORIA_DISTANCIA'].value_counts().sort_index()
        for cat, count in dist_dist.items():
            pct = (count / len(df_final)) * 100
            print(f'   {cat:15s}: {count:6,} ({pct:5.2f}%)')
    print('\n' + '='*70)
    print('='*70)

## Pipeline Procesamiento

In [ ]:
class PipelineProcesamiento:
    def __init__(self):
        self.transformadores = []
        self.df_datos = None
    def cargar_datos(self, ruta: str):
        self.df_datos = pd.read_csv(ruta)
        return self
    def agregar_transformador(self, t):
        self.transformadores.append(t)
        return self
    def ejecutar(self):
        df = self.df_datos.copy()
        for t in self.transformadores:
            df = t.ejecutar(df)
        return df

## **PRUEBA COMPLETA DEL SISTEMA**

Ejecuta el pipeline completo con el dataset real para validar toda la funcionalidad.

In [ ]:
# FUNCION PRINCIPAL: Ejecutar Pipeline Completo
def ejecutar_pipeline_completo(ruta_dataset, verbose=False):
    """
    Ejecuta el pipeline completo de transformacion de datos de terremotos.
    
    Parametros:
        ruta_dataset (str): Ruta al archivo CSV
        verbose (bool): Si True, muestra detalles de cada transformacion
    
    Retorna:
        DataFrame: Dataset completamente procesado
    """
    print("\n" + "="*100)
    print("EJECUTANDO PIPELINE COMPLETO")
    print("="*100 + "\n")
    
    # Paso 1: Cargar datos
    print("PASO 1: Cargando datos del dataset...")
    cargador = CargadorDatos(ruta_dataset)
    df = cargador.ejecutar(pd.DataFrame())
    print(f"   OK - {len(df):,} registros cargados\n")
    
    # Lista de transformadores a aplicar en orden
    transformadores = [
        ("Limpieza de duplicados", LimpiezaDuplicados()),
        ("Renombrado de variables", RenombradoVariables()),
        ("Extraccion de fechas UTC", ProcesadorFechaUTC()),
        ("Separacion de profundidad", SeparadorProfundidad()),
        ("Separacion de magnitud", SeparadorMagnitud()),
        ("Separacion de ubicacion", SeparadorUbicacion()),
        ("Estandarizacion de magnitud MW", EstandarizadorMagnitudMW()),
        ("Filtro geografico (Chile)", FiltroGeograficoChile()),
        ("Tratamiento de valores nulos", TratamientoNulos(estrategia='eliminar')),
    ]
    
    # Paso 2: Aplicar transformadores
    print("PASO 2: Aplicando transformadores...")
    for i, (nombre, transformador) in enumerate(transformadores, 1):
        filas_antes = len(df)
        
        if verbose:
            print(f"\n   {i}. {nombre}...")
            df = ejecutar_transformador_con_log(transformador, df, nombre_prueba=nombre)
        else:
            try:
                df = transformador.ejecutar(df)
                filas_despues = len(df)
                cambio = filas_despues - filas_antes
                print(f"   {i}. {nombre}: {filas_antes:,} -> {filas_despues:,} ({'+' if cambio > 0 else ''}{cambio:,})")
            except Exception as e:
                print(f"   ERROR en {nombre}: {e}")
                raise
    
    # Paso 3: Aplicar categorizacion final
    print(f"\nPASO 3: Aplicando categorizacion final...")
    categorizador = CategorizadorDatos()
    
    # Registrar categorias
    categorizador.registrar_categoria(
        'PROF_CAT',
        'PROFUNDIDAD_VALOR',
        [0, 70, 300, float('inf')],
        ['somero', 'intermedio', 'profundo'],
        'Clasificacion de profundidad sismica'
    )
    
    categorizador.registrar_categoria(
        'MAG_CAT',
        'MAGNITUD_MW',
        [0, 2, 4, 5, 6, 7, 8, float('inf')],
        ['micro', 'menor', 'ligero', 'moderado', 'fuerte', 'mayor', 'grande'],
        'Clasificacion de magnitud Mw'
    )
    
    categorizador.registrar_categoria(
        'DIST_CAT',
        'DISTANCIA_EPICENTRO_KM',
        [0, 50, 100, 200, float('inf')],
        ['muy_cercano', 'cercano', 'intermedio', 'lejano'],
        'Clasificacion de distancia al epicentro'
    )
    
    if verbose:
        df = ejecutar_transformador_con_log(categorizador, df, nombre_prueba="Categorizacion de datos")
    else:
        df = categorizador.ejecutar(df)
        print(f"   OK - Categorias aplicadas: {categorizador.get_categorias_registradas()}")
    
    print(f"\nPipeline completado exitosamente")
    print("="*100 + "\n")
    
    return df

## **▶️ EJECUTAR PRUEBA COMPLETA DEL SISTEMA**

Ejecuta esta celda para procesar el dataset completo usando todo el pipeline POO

In [84]:
# ============================================================================
# CARGAR TODAS LAS CLASES NECESARIAS PARA LA PRUEBA
# ============================================================================
print("Cargando todas las definiciones de clases...")

# Clase 1: LimpiezaDuplicados
class LimpiezaDuplicados(Transformer):
    def __init__(self):
        super().__init__("LimpiezaDuplicados", "Elimina filas duplicadas")
        self._duplicados_eliminados = 0
    def validar_entrada(self, df):
        return not df.empty
    def transformar(self, df):
        filas_iniciales = len(df)
        df_limpio = df.drop_duplicates().copy()
        self._duplicados_eliminados = filas_iniciales - len(df_limpio)
        self._logger.info(f"Duplicados eliminados: {self._duplicados_eliminados}")
        return df_limpio

# Clase 2: RenombradoVariables
class RenombradoVariables(Transformer):
    MAPEO_ESTANDAR = {
        'UTC Date': 'FECHA_UTC',
        'Profoundity': 'PROFUNDIDAD',
        'Magnitude': 'MAGNITUD',
        'Date': 'FECHA_LOCAL',
        'Hour': 'HORA_LOCAL',
        'Location': 'UBICACION',
        'Latitude': 'LATITUD',
        'Longitude': 'LONGITUD'
    }
    def __init__(self, mapeo=None):
        super().__init__("RenombradoVariables", "Estandariza nombres de columnas")
        self.mapeo = mapeo or self.MAPEO_ESTANDAR
    def validar_entrada(self, df):
        return not df.empty
    def transformar(self, df):
        df_renombrado = df.rename(columns=self.mapeo).copy()
        self._logger.info(f"Variables renombradas: {list(self.mapeo.values())}")
        return df_renombrado

# Clase 3: ProcesadorFechaUTC
class ProcesadorFechaUTC(Transformer):
    def __init__(self):
        super().__init__("ProcesadorFechaUTC", "Extrae componentes de fecha UTC")
    def validar_entrada(self, df):
        return 'FECHA_UTC' in df.columns
    def transformar(self, df):
        df_copia = df.copy()
        df_copia['FECHA_UTC'] = pd.to_datetime(df_copia['FECHA_UTC'], errors='coerce')
        df_copia['ANIO_UTC'] = df_copia['FECHA_UTC'].dt.year
        df_copia['MES_UTC'] = df_copia['FECHA_UTC'].dt.month
        df_copia['DIA_UTC'] = df_copia['FECHA_UTC'].dt.day
        self._logger.info("Fechas UTC procesadas")
        return df_copia

# Clase 4: SeparadorProfundidad
class SeparadorProfundidad(Transformer):
    def __init__(self):
        super().__init__("SeparadorProfundidad", "Separa valor y unidad de profundidad")
    def validar_entrada(self, df):
        return 'PROFUNDIDAD' in df.columns
    def transformar(self, df):
        df_copia = df.copy()
        patron = r'([0-9.]+)\s*([a-zA-Z]+)'
        df_copia[['PROFUNDIDAD_VALOR', 'PROFUNDIDAD_UNIDAD']] = df_copia['PROFUNDIDAD'].str.extract(patron)
        df_copia['PROFUNDIDAD_VALOR'] = pd.to_numeric(df_copia['PROFUNDIDAD_VALOR'], errors='coerce')
        return df_copia

# Clase 5: SeparadorMagnitud
class SeparadorMagnitud(Transformer):
    def __init__(self):
        super().__init__("SeparadorMagnitud", "Separa valor y escala de magnitud")
    def validar_entrada(self, df):
        return 'MAGNITUD' in df.columns
    def transformar(self, df):
        df_copia = df.copy()
        patron = r'([0-9.]+)\s*([A-Za-z]+)'
        df_copia[['MAGNITUD_VALOR', 'MAGNITUD_ESCALA']] = df_copia['MAGNITUD'].str.extract(patron)
        df_copia['MAGNITUD_VALOR'] = pd.to_numeric(df_copia['MAGNITUD_VALOR'], errors='coerce')
        return df_copia

# Clase 6: SeparadorUbicacion
class SeparadorUbicacion(Transformer):
    def __init__(self):
        super().__init__("SeparadorUbicacion", "Separa componentes de ubicacion")
    def validar_entrada(self, df):
        return 'UBICACION' in df.columns
    def transformar(self, df):
        df_copia = df.copy()
        patron1 = r'([0-9]+)\s*km\s*al\s*(\w+)\s*de\s*(.+?)(?:\s*-|$)'
        patron2 = r'([0-9]+)\s*km\s*de\s*(.+?)$'
        extracciones = df_copia['UBICACION'].str.extract(patron1)
        mask_vacia = extracciones[0].isna()
        extracciones.loc[mask_vacia] = df_copia.loc[mask_vacia, 'UBICACION'].str.extract(patron2)
        df_copia['DISTANCIA_EPICENTRO_KM'] = pd.to_numeric(extracciones[0], errors='coerce')
        df_copia['DIRECCION'] = extracciones[1]
        df_copia['NOMBRE_LUGAR'] = extracciones[2]
        return df_copia

# Clase 7: EstandarizadorMagnitudMW
class EstandarizadorMagnitudMW(Transformer):
    def __init__(self):
        super().__init__("EstandarizadorMagnitudMW", "Convierte magnitudes a Mw")
        self.conversiones = {
            'Ml': lambda x: 0.85 * x + 1.03,
            'Mlv': lambda x: 0.85 * x + 1.03,
            'Mb': lambda x: 0.85 * x + 1.03,
            'Mc': lambda x: 0.85 * x + 1.03,
            'Ms': lambda x: 0.67 * x + 2.07,
        }
    def validar_entrada(self, df):
        return 'MAGNITUD_VALOR' in df.columns and 'MAGNITUD_ESCALA' in df.columns
    def transformar(self, df):
        df_copia = df.copy()
        df_copia['MAGNITUD_MW'] = df_copia['MAGNITUD_VALOR']
        df_copia['MAGNITUD_ESCALA'] = df_copia['MAGNITUD_ESCALA'].astype(str)
        mlv_count = (df_copia['MAGNITUD_ESCALA'] == 'Mlv').sum()
        df_copia.loc[df_copia['MAGNITUD_ESCALA'] == 'Mlv', 'MAGNITUD_ESCALA'] = 'Ml'
        for escala, formula in self.conversiones.items():
            mask = df_copia['MAGNITUD_ESCALA'] == escala
            count = mask.sum()
            if count > 0:
                df_copia.loc[mask, 'MAGNITUD_MW'] = formula(df_copia.loc[mask, 'MAGNITUD_VALOR'])
        return df_copia

# Clase 8: FiltroGeograficoChile
class FiltroGeograficoChile(Transformer):
    def __init__(self, lat_min=-56.0, lat_max=-17.0, lon_min=-90.0, lon_max=-66.0):
        super().__init__("FiltroGeograficoChile", "Filtra eventos en Chile")
        self.lat_min = lat_min
        self.lat_max = lat_max
        self.lon_min = lon_min
        self.lon_max = lon_max
    def validar_entrada(self, df):
        return 'LATITUD' in df.columns and 'LONGITUD' in df.columns
    def transformar(self, df):
        mask_chile = (
            (df['LATITUD'] >= self.lat_min) & (df['LATITUD'] <= self.lat_max) &
            (df['LONGITUD'] >= self.lon_min) & (df['LONGITUD'] <= self.lon_max)
        )
        df_filtrado = df[mask_chile].copy()
        return df_filtrado

# Clase 9: TratamientoNulos
class TratamientoNulos(Transformer):
    def __init__(self, estrategia='eliminar', columnas=None):
        super().__init__("TratamientoNulos", "Maneja valores nulos")
        self.estrategia = estrategia
        self.columnas = columnas
    def validar_entrada(self, df):
        return not df.empty
    def transformar(self, df):
        if self.estrategia == 'eliminar':
            if self.columnas:
                df_limpio = df.dropna(subset=self.columnas).copy()
            else:
                df_limpio = df.dropna().copy()
        else:
            df_limpio = df.copy()
        return df_limpio

# Clase 10: CategorizadorDatos
class CategorizadorDatos(Transformer):
    def __init__(self):
        super().__init__("CategorizadorDatos", "Categoriza datos")
        self._categorias = {}
    def registrar_categoria(self, nombre, columna, bins, etiquetas, descripcion=""):
        self._categorias[nombre] = {
            'columna': columna,
            'bins': bins,
            'etiquetas': etiquetas,
            'descripcion': descripcion
        }
    def get_categorias_registradas(self):
        return list(self._categorias.keys())
    def validar_entrada(self, df):
        return not df.empty
    def transformar(self, df):
        df_copia = df.copy()
        for nombre, config in self._categorias.items():
            if config['columna'] in df_copia.columns:
                df_copia[nombre] = pd.cut(df_copia[config['columna']], bins=config['bins'], labels=config['etiquetas'])
        return df_copia

print("OK - Todas las clases cargadas exitosamente")

Cargando todas las definiciones de clases...
OK - Todas las clases cargadas exitosamente


In [85]:
# ============================================================================
# PRUEBA REAL DEL PIPELINE COMPLETO - PROCESAMIENTO COMPLETO
# ============================================================================
print("\n" + "="*100)
print("INICIANDO PRUEBA COMPLETA DEL SISTEMA POO")
print("="*100 + "\n")

try:
    print("Procesando dataset completo. Esto puede tomar algunos segundos...\n")
    df_final = ejecutar_pipeline_completo(RUTA_DATASET, verbose=False)
    
    # ========== MOSTRAR RESULTADOS ==========
    print("\n" + "="*100)
    print("PIPELINE EJECUTADO EXITOSAMENTE")
    print("="*100)
    
    print(f"\nRESUMEN DEL DATASET PROCESADO:")
    print(f"   Total de registros: {len(df_final):,}")
    print(f"   Total de columnas: {len(df_final.columns)}")
    print(f"   Memoria utilizada: {df_final.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    print(f"\nRANGO DE DATOS:")
    if 'PROFUNDIDAD_VALOR' in df_final.columns:
        print(f"   Profundidad: {df_final['PROFUNDIDAD_VALOR'].min():.1f} - {df_final['PROFUNDIDAD_VALOR'].max():.1f} km")
    if 'MAGNITUD_MW' in df_final.columns:
        print(f"   Magnitud MW: {df_final['MAGNITUD_MW'].min():.2f} - {df_final['MAGNITUD_MW'].max():.2f}")
    if 'LATITUD' in df_final.columns:
        print(f"   Latitud: {df_final['LATITUD'].min():.2f} a {df_final['LATITUD'].max():.2f}")
    if 'LONGITUD' in df_final.columns:
        print(f"   Longitud: {df_final['LONGITUD'].min():.2f} a {df_final['LONGITUD'].max():.2f}")
    
    print(f"\nCATEGORIAS GENERADAS:")
    if 'PROF_CAT' in df_final.columns:
        print(f"   Profundidad sísmica:")
        for cat, count in df_final['PROF_CAT'].value_counts().sort_index().items():
            print(f"      - {cat}: {count:,} registros")
    
    if 'MAG_CAT' in df_final.columns:
        print(f"   Magnitud Mw:")
        for cat, count in df_final['MAG_CAT'].value_counts().items():
            print(f"      - {cat}: {count:,} registros")
    
    if 'DIST_CAT' in df_final.columns:
        print(f"   Distancia al epicentro:")
        for cat, count in df_final['DIST_CAT'].value_counts().sort_index().items():
            print(f"      - {cat}: {count:,} registros")
    
    print(f"\nVISTA PREVIA (Primeras 5 filas):")
    print(df_final.head(5).to_string())
    
    print(f"\nINFORMACION DE COLUMNAS:")
    print(f"Columnas generadas: {list(df_final.columns)}")
    
    print(f"\nDATA LISTA PARA ANALISIS Y MODELADO")
    print("="*100 + "\n")
    
except FileNotFoundError as e:
    print(f"\nERROR: No se encontro el archivo del dataset")
    print(f"   Ruta esperada: {RUTA_DATASET}")
    print(f"   Error: {e}")
    
except Exception as e:
    print(f"\nERROR DURANTE LA EJECUCION: {str(e)}")
    print("\nDetalles del error:")
    import traceback
    traceback.print_exc()

2026-06-19 16:28:28,270 - INFO - Iniciando: CargadorDatos



INICIANDO PRUEBA COMPLETA DEL SISTEMA POO

Procesando dataset completo. Esto puede tomar algunos segundos...


▶️  EJECUTANDO PIPELINE COMPLETO

📥 PASO 1: Cargando datos del dataset...


2026-06-19 16:28:28,623 - INFO - Cargadas 134062 filas, 8 columnas
2026-06-19 16:28:28,624 - INFO - Completado en 0.3541s
2026-06-19 16:28:28,625 - INFO - Iniciando: LimpiezaDuplicados


   ✓ 134,062 registros cargados

⚙️  PASO 2: Aplicando transformadores...


2026-06-19 16:28:28,905 - INFO - Duplicados eliminados: 3
2026-06-19 16:28:28,906 - INFO - Completado en 0.2816s
2026-06-19 16:28:28,912 - INFO - Iniciando: RenombradoVariables
2026-06-19 16:28:28,925 - INFO - Variables renombradas: ['FECHA_UTC', 'PROFUNDIDAD', 'MAGNITUD', 'FECHA_LOCAL', 'HORA_LOCAL', 'UBICACION', 'LATITUD', 'LONGITUD']
2026-06-19 16:28:28,926 - INFO - Completado en 0.0144s
2026-06-19 16:28:28,941 - INFO - Iniciando: ProcesadorFechaUTC
2026-06-19 16:28:29,071 - INFO - Fechas UTC procesadas
2026-06-19 16:28:29,072 - INFO - Completado en 0.1305s
2026-06-19 16:28:29,082 - INFO - Iniciando: SeparadorProfundidad


...

   5. Separación de magnitud: 134,059 → 134,059 (0)


2026-06-19 16:28:30,107 - INFO - Completado en 0.4034s
2026-06-19 16:28:30,116 - INFO - Iniciando: EstandarizadorMagnitudMW
2026-06-19 16:28:30,258 - INFO - Completado en 0.1420s
2026-06-19 16:28:30,272 - INFO - Iniciando: FiltroGeograficoChile


   6. Separación de ubicación: 134,059 → 134,059 (0)
   7. Estandarización de magnitud MW: 134,059 → 134,059 (0)


2026-06-19 16:28:30,374 - INFO - Completado en 0.1022s
2026-06-19 16:28:30,393 - INFO - Iniciando: TratamientoNulos
2026-06-19 16:28:30,475 - INFO - Completado en 0.0815s
2026-06-19 16:28:30,490 - INFO - Iniciando: CategorizadorDatos
2026-06-19 16:28:30,557 - INFO - Completado en 0.0676s


   8. Filtro geográfico (Chile): 134,059 → 132,642 (-1,417)
   9. Tratamiento de valores nulos: 132,642 → 132,642 (0)

📊 PASO 3: Aplicando categorización final...
   ✓ Categorías aplicadas: ['PROF_CAT', 'MAG_CAT', 'DIST_CAT']

✅ Pipeline completado exitosamente


PIPELINE EJECUTADO EXITOSAMENTE

RESUMEN DEL DATASET PROCESADO:
   Total de registros: 132,642
   Total de columnas: 22
   Memoria utilizada: 76.67 MB

RANGO DE DATOS:
   Profundidad: 0.0 - 407.0 km
   Magnitud MW: 1.88 - 8.80
   Latitud: -54.84 a -17.00
   Longitud: -89.09 a -66.03

CATEGORIAS GENERADAS:
   Profundidad sísmica:
      - somero: 71,701 registros
      - intermedio: 60,370 registros
      - profundo: 55 registros
   Magnitud Mw:
      - menor: 89,769 registros
      - ligero: 39,239 registros
      - moderado: 3,425 registros
      - fuerte: 197 registros
      - mayor: 6 registros
      - micro: 3 registros
      - grande: 3 registros
   Distancia al epicentro:
      - muy_cercano: 84,510 registros
      - cerc

In [ ]:
# ============================================================================
# ANÁLISIS D: EFICIENCIA Y OPTIMIZACIÓN - MEDICIONES CON TIMEIT
# ============================================================================
import timeit
from collections import defaultdict

print("\n" + "="*100)
print("ANÁLISIS D: EFICIENCIA Y OPTIMIZACIÓN")
print("="*100)

# 1. Medición de tiempo de cada transformador (múltiples ejecuciones)
print("\n1. MEDICIONES DE PERFORMANCE (timeit - 5 ejecuciones)")
print("-" * 100)

# Cargar datos una sola vez
cargador = CargadorDatos(RUTA_DATASET)
df_base = cargador.transformar(pd.DataFrame())

tiempos_transformadores = {}

# Transformador 1: LimpiezaDuplicados
tiempos = []
for _ in range(5):
    limpiador = LimpiezaDuplicados()
    t = timeit.timeit(lambda: limpiador.transformar(df_base.copy()), number=1)
    tiempos.append(t)
tiempos_transformadores['LimpiezaDuplicados'] = {
    'min': min(tiempos),
    'max': max(tiempos),
    'promedio': sum(tiempos)/len(tiempos),
    'operación': 'df.drop_duplicates()'
}
print(f"LimpiezaDuplicados:")
print(f"  Tiempo promedio: {tiempos_transformadores['LimpiezaDuplicados']['promedio']*1000:.3f}ms")
print(f"  Rango: {tiempos_transformadores['LimpiezaDuplicados']['min']*1000:.3f}ms - {tiempos_transformadores['LimpiezaDuplicados']['max']*1000:.3f}ms")
print(f"  Complejidad: O(n log n)")

# Transformador 2: RenombradoVariables
tiempos = []
for _ in range(5):
    renombrador = RenombradoVariables()
    t = timeit.timeit(lambda: renombrador.transformar(df_base.copy()), number=1)
    tiempos.append(t)
tiempos_transformadores['RenombradoVariables'] = {
    'min': min(tiempos),
    'max': max(tiempos),
    'promedio': sum(tiempos)/len(tiempos),
    'operación': 'df.rename(columns=dict)'
}
print(f"\nRenombradoVariables:")
print(f"  Tiempo promedio: {tiempos_transformadores['RenombradoVariables']['promedio']*1000:.3f}ms")
print(f"  Rango: {tiempos_transformadores['RenombradoVariables']['min']*1000:.3f}ms - {tiempos_transformadores['RenombradoVariables']['max']*1000:.3f}ms")
print(f"  Complejidad: O(1)")

print("\n" + "="*100 + "\n")